<div style="display: flex; align-items: center; gap: 20px; background-color: rgba(173, 38, 191, 0.74); padding: 20px; border-radius: 10px; color: white;">
  <img src="https://gna.org.co/wp-content/uploads/2025/06/LOGO-GNA-TRANSP.png" alt="GNA logo" style="height: 120px;">
  <div>
      <h1 style="margin: 0; font-size: 2.3em; font-weight: bold;">Workshop: fMRI basics</h1>
      <p style="margin: 10px 0 5px 0; font-size: 1.3em;"><strong>Grupo de Neurociencias de Antioquia</strong></p>
      <p style="margin: 10px 0 5px 0; font-size: 1.2em;"><strong>Línea de Neurociencias Computacionales</strong> </p>
      <p style="margin: 10px 0 5px 0; font-size: 1em;">Prepared by: Keveen Rodríguez, Luisa Fernanda Taho </p>
      <p style="margin: 10px 0 5px 0; font-size: 1em;">Revised by: Johanna Roca </p>
      <p style="margin: 5px 0; font-size: 1em;">Date:26/08/26  </p>
  </div>
</div>

## **Configuración de rutas del proyecto** ≽(•⩊ •マ≼

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.io_utils import preparar_rutas, inventariar_t1w
import importlib
import src.func_preproc as func


## **Inventario rápido** 𑣲⋆

In [ ]:
PATHS = preparar_rutas()
inventario_t1 = inventariar_t1w()

display(inventario_t1.drop(columns="ruta"))

## **Preprocesamiento anatómico** (✿ᴗ͈ˬᴗ͈)⁾⁾

## 1.1 Extracción cerebral del T1w

Se utiliza una U-Net 3D de ANTsPyNet entrenada para imágenes T1w. La red devuelve una máscara probabilística, que se binariza y se refina conservando el componente conexo principal y rellenando huecos. En esta etapa no se aplica N4.

In [ ]:
from src.anat_preproc import (
    ALGORITMO,
    extraer_craneo_lote,
    mostrar_entrada_salida,
)

OUTPUT_ROOT = PATHS["derivatives"] / "anat_preproc"

print("Algoritmo:", ALGORITMO)

resultados_extraccion = extraer_craneo_lote(inventario_t1,OUTPUT_ROOT,overwrite=True,)

resumen = resultados_extraccion.assign(
    entrada=resultados_extraccion["input_path"].map(lambda ruta: ruta.name),
    salida=resultados_extraccion["brain_path"].map(lambda ruta: ruta.name),
)

display(resumen[["subject", "entrada", "salida"]])


In [ ]:
mostrar_entrada_salida(resultados_extraccion)

## 1.2 Segmentación de tejidos

In [ ]:
from src.anat_preproc import (
    ALGORITMO_SEGMENTACION,
    mostrar_segmentacion,
    segmentar_tejidos_lote,
)

In [ ]:
print("Algoritmo:", ALGORITMO_SEGMENTACION)

resultados_segmentacion = segmentar_tejidos_lote(
    resultados_extraccion,
    overwrite=True,
)

mostrar_segmentacion(resultados_segmentacion)

## 1.3 Normalización a MNI

In [ ]:
from src.anat_preproc import (
    ALGORITMO_NORMALIZACION,
    mostrar_normalizacion,
    normalizar_t1_lote,
    preparar_plantilla_mni,
)

MNI_TEMPLATE = preparar_plantilla_mni(
    PATHS["external"] / "templates"
)

print("Algoritmo:", ALGORITMO_NORMALIZACION)
print("Referencia:", MNI_TEMPLATE.name)

resultados_normalizacion = normalizar_t1_lote(
    resultados_segmentacion,
    MNI_TEMPLATE,
    overwrite=True,
)

mostrar_normalizacion(resultados_normalizacion)

## **2. Preprocesamiento funcional** ₍ᐢ._.ᐢ₎♡ ༘

### 2.1 Inventario funcional

In [ ]:
from src.func_preproc import (
    ALGORITMO_INVENTARIO,
    inventariar_bold,
    mostrar_qc_bold_crudo,
)

print("Algoritmo:", ALGORITMO_INVENTARIO)

inventario_bold = inventariar_bold(
    PATHS["bids"]
)

display(
    inventario_bold.drop(
        columns=["ruta", "json_path"]
    )
)

In [ ]:
mostrar_qc_bold_crudo(
    inventario_bold
)

## 2.2 Descarte de volúmenes iniciales

In [ ]:

OUTPUT_FUNC = PATHS["derivatives"] / "func_preproc"
N_DESCARTAR = 5

print("Algoritmo:", func.ALGORITMO_DESCARTE)
print("Volúmenes iniciales a descartar:", N_DESCARTAR)

resultados_descarte = func.descartar_volumenes_lote(
    inventario=inventario_bold,
    output_root=OUTPUT_FUNC,
    n_descartar=N_DESCARTAR,
    overwrite=True,
)

display(
    resultados_descarte[
        [
            "subject",
            "session",
            "original_volumes",
            "discarded_volumes",
            "remaining_volumes",
            "duration_min",
            "status",
        ]
    ]
)

In [ ]:
func.mostrar_qc_descarte(
    resultados_descarte
)

## 2.3 Corrección por movimiento

In [ ]:
UMBRAL_FD = 0.5

print("Algoritmo:", func.ALGORITMO_MOVIMIENTO)
print("Umbral de FD:", UMBRAL_FD, "mm")

resultados_movimiento = func.corregir_movimiento_lote(
    resultados_descarte,
    umbral_fd=UMBRAL_FD,
    overwrite=True,
    verbose=True,
)

display(
    resultados_movimiento[
        [
            "subject",
            "session",
            "status",
            "seconds",
            "fd_mean_mm",
            "fd_max_mm",
            "fd_above_threshold",
            "fd_above_percent",
        ]
    ]
)

## 2.3.1 QC del MoCo

Ojooo: Este bloque tarda aproximadamente 15 min en correr, **por favor no correrlo durante el workshop.**

In [ ]:
import importlib
import src.func_preproc as func

importlib.reload(func)

print("Algoritmo:", func.ALGORITMO_QC_MOVIMIENTO)

resultados_movimiento = func.estimar_movimiento_residual_lote(
    resultados_movimiento,
    overwrite=False,
    verbose=False,
)

resumen_qc_movimiento = func.mostrar_qc_movimiento(
    resultados_movimiento,
    umbral_fd=0.5,
    incluir_senal=False,
)

## 2.4 Slice Timing Correction 

In [ ]:
print("Algoritmo:", func.ALGORITMO_SLICE_TIMING)

resultados_stc = func.corregir_slice_timing_lote(
    resultados_movimiento,
    inventario_bold,
    referencia="mitad_TR",
    orden_spline=3,
    overwrite=False,
)

display(
    resultados_stc[
        [
            "subject",
            "stc_status",
            "stc_seconds",
            "reference_time_s",
            "max_slice_shift_volumes",
            "stc_path",
        ]
    ]
)

In [ ]:
func.mostrar_qc_slice_timing(resultados_stc)

## 2.5 Coregistro con T1

In [ ]:
print("Algoritmo:", func.ALGORITMO_CORREGISTRO)

resultados_coregistro = func.coregistrar_bold_t1_lote(
    resultados_stc,
    resultados_segmentacion,
    overwrite=False,
    verbose=False,
)

display(
    resultados_coregistro[
        [
            "subject",
            "coreg_status",
            "coreg_seconds",
            "dice_epi_t1",
            "epi_inside_t1_pct",
            "t1_covered_by_epi_pct",
            "bold_native_path",
            "boldref_t1_path",
        ]
    ]
)

In [ ]:
func.mostrar_qc_coregistro(resultados_coregistro)

## 2.6 Normalización

In [ ]:
print("Algoritmo:", func.ALGORITMO_NORMALIZACION_BOLD)

resultados_bold_mni = func.normalizar_bold_mni_lote(
    resultados_coregistro,
    resultados_normalizacion,
    overwrite=False,
)

display(
    resultados_bold_mni[
        [
            "subject",
            "normalization_status",
            "normalization_seconds",
            "dice_epi_mni",
            "epi_inside_mni_pct",
            "mni_covered_by_epi_pct",
            "normalized_bold_path",
        ]
    ]
)

In [ ]:
func.mostrar_qc_normalizacion_bold(
    resultados_bold_mni
)

## 2.7 aCompCor

In [ ]:
print("Algoritmo:", func.ALGORITMO_ACOMPCOR)

resultados_acompcor = func.extraer_acompcor_lote(
    resultados_bold_mni,
    resultados_segmentacion,
    resultados_normalizacion,
    n_components=5,
    probability_threshold=0.80,
    erosion_iterations=1,
    overwrite=False,
)

display(
    resultados_acompcor[
        [
            "subject",
            "acompcor_status",
            "acompcor_seconds",
            "wm_mask_voxels",
            "csf_mask_voxels",
            "wm_variance_first_n_pct",
            "csf_variance_first_n_pct",
            "acompcor_path",
        ]
    ]
)

In [ ]:
func.mostrar_qc_acompcor(
    resultados_acompcor
)

## 2.8 Denoising

In [ ]:
print("Algoritmo:", func.ALGORITMO_DENOISING)

resultados_denoising = func.aplicar_denoising_lote(
    resultados_acompcor,
    high_pass=0.008,
    low_pass=0.09,
    fd_threshold=0.5,
    gschange_threshold=3.0,
    n_acompcor=5,
    fast_mode=True,
    overwrite=True,
)

display(
    resultados_denoising[
        [
            "subject",
            "denoising_status",
            "denoising_seconds",
            "retained_volumes",
            "censored_volumes",
            "effective_dof",
            "corr_fd_dvars_before",
            "corr_fd_dvars_after",
            "denoised_path",
        ]
    ]
)

In [ ]:
func.mostrar_qc_denoising(
    resultados_denoising
)

In [ ]:
import importlib
import src.func_preproc as func

importlib.reload(func)

## 3. **Extracción de la señal BOLD - Schaefer 400** ૮ ˶ᵔ ᵕ ᵔ˶ ა

In [ ]:
print("Algoritmo:", func.ALGORITMO_PARCELACION)

ATLAS_SCHAEFER = func.preparar_atlas_schaefer(
    atlas_root=PATHS["external"] / "atlases",
    template_path=MNI_TEMPLATE,
    n_rois=400,
    yeo_networks=17,
    resolution_mm=2,
)

print("Atlas:", ATLAS_SCHAEFER["name"])

resultados_roi, cobertura_roi = func.extraer_series_roi_lote(
    resultados_denoising,
    ATLAS_SCHAEFER,
    min_coverage=0.50,
    min_voxels=10,
    overwrite=False,
)

display(
    resultados_roi[
        [
            "subject",
            "roi_status",
            "roi_seconds",
            "valid_rois",
            "common_valid_rois",
            "mean_roi_coverage_pct",
            "roi_timeseries_path",
        ]
    ]
)

In [ ]:
func.mostrar_qc_series_roi(
    resultados_roi,
    cobertura_roi,
    ATLAS_SCHAEFER,
)

In [ ]:
display(
    resultados_roi[
        [
            "subject",
            "valid_rois",
            "common_valid_rois",
            "mean_roi_coverage_pct",
        ]
    ]
)

In [ ]:
rois_excluidas = (
    cobertura_roi.loc[
        ~cobertura_roi["common_valid"],
        [
            "subject",
            "roi_id",
            "label",
            "network",
            "coverage_pct",
            "valid",
        ],
    ]
    .sort_values(["roi_id", "subject"])
)

display(rois_excluidas)

## 4. **Conectividad funcional** ૮ • ﻌ - ა

In [ ]:

print("Algoritmo:", func.ALGORITMO_CONECTIVIDAD)

resultados_conectividad = func.calcular_conectividad_lote(
    resultados_roi,
    cobertura_roi,
    ATLAS_SCHAEFER,
    overwrite=False,
)

display(
    resultados_conectividad[
        [
            "subject",
            "connectivity_status",
            "connectivity_rois",
            "mean_absolute_r",
            "connectivity_qc_valid",
            "connectivity_path",
        ]
    ]
)

In [ ]:
func.mostrar_qc_conectividad(
    resultados_conectividad,
    ATLAS_SCHAEFER,
)

# **¡Gracias!**

# ฅ^>⩊<^ ฅ